# Correlation Analysis

Cross-domain correlation matrix (Spearman). Load two or more domains,
merge on a shared key, select numeric columns, and render a heatmap.

**Usage:**
1. Load the relevant DataFrames in Section 1.
2. Merge and select numeric columns in Section 2.
3. Run Section 3 for the heatmap.

In [ ]:
# Section 1 — Setup & Data Loading
from utils.db import load_handover, load_participants, load_questionnaire_responses
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

df_h = load_handover()
df_p = load_participants()

In [ ]:
# Section 2 — Data Preparation & Merge

# Derive handover duration
for col in ['giver_grasped_object', 'giver_released_object']:
    df_h[col] = pd.to_datetime(df_h[col], errors='coerce')
df_h['duration_ms'] = (
    df_h['giver_released_object'] - df_h['giver_grasped_object']
).dt.total_seconds() * 1000

# Merge handover with participant data on the giver role
merged = df_h.merge(
    df_p[['participant_id', 'age']],
    left_on='giver',
    right_on='participant_id',
    how='left'
)

# Select numeric columns for correlation
# Add or remove columns as needed
numeric_cols = ['duration_ms', 'age']
df_corr = merged[numeric_cols].dropna()
print(f'Rows available for correlation: {len(df_corr)}')

In [ ]:
# Section 3 — Correlation Heatmap (Spearman)
corr_matrix = df_corr.corr(method='spearman')
print(corr_matrix)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Spearman Correlation Matrix')
plt.tight_layout()
plt.show()